In [ ]:
import json
import time
import os
from pathlib import Path
import kaggle_benchmarks as kbench

In [ ]:
# Helpers
def normalize_segment_text(text: str) -> str:
    """Đảm bảo segment kết thúc bằng dấu câu rõ ràng trước khi join bằng
    dấu cách với segment kế tiếp, tránh 2 câu dính liền không có dấu ngăn cách.
    Chỉ thêm "." khi segment CHƯA kết thúc bằng bất kỳ dấu câu nào (áp dụng
    SAU khi đã fix chính tả/dấu câu qua LLM, nên hầu hết segment đã có dấu
    đúng, hàm này chỉ xử lý các trường hợp còn thiếu)."""
    text = text.strip()
    if not text:
        return text
    END_PUNCTS = ('.', '?', '!', ';', ':', '…', ',')
    if not text.endswith(END_PUNCTS):
        text += "."
    return text


def build_multi_prompt(texts):
    payload_text = "\n|\n".join([t.strip() for t in texts])
    return f"""Bạn là chuyên gia biên tập. Hãy sửa lỗi chính tả, lỗi Telex/VNI cho các đoạn tin (event) phân cách bởi dấu '|' dưới đây.
YÊU CẦU:
- Chỉ sửa lỗi chính tả và dấu câu (nếu cần).
- Loại bỏ các cụm từ trùng lặp (Ví dụ: "PHÔNG PHÔNG PHÔNG...").
- TUYỆT ĐỐI KHÔNG sửa tên riêng gốc nước ngoài thành các từ Việt hóa như "Renan Gôm"
- BẮT BUỘC trả về đúng {len(texts)} câu, ngăn cách bằng dấu '|'.
- TUYỆT ĐỐI KHÔNG thêm bất kỳ lời dẫn, lời chào, lời giải thích nào (VD: "Dưới đây là danh sách các đoạn tin đã được sửa lỗi chính tả, lỗi Telex/VNI, loại bỏ cụm từ trùng lặp và chuẩn hóa dấu câu theo đúng yêu cầu của bạn: "). Chỉ trả về đúng nội dung {len(texts)} câu nối bằng dấu '|', không có gì khác trước hay sau.
 
Danh sách câu:
{payload_text}"""


def build_single_prompt(text):
    # 🔧 Dùng riêng cho trường hợp chỉ có 1 đoạn văn (thường là 1 event dài,
    # gồm nhiều câu bên trong). KHÔNG dùng prompt kiểu tách '|' ở trên, vì khi
    # đưa vào 1 khối văn bản nhiều câu, AI sẽ tự tách thành nhiều câu trả về
    # (vd 8 câu) dù chỉ có 1 phần tử input -> gây lệch validate số lượng.
    # Prompt này yêu cầu giữ nguyên là 1 khối liền mạch, không tách gì cả.
    return f"""Bạn là chuyên gia biên tập. Hãy sửa lỗi chính tả, lỗi Telex/VNI cho đoạn văn dưới đây.
YÊU CẦU:
- Chỉ sửa lỗi chính tả và dấu câu (nếu cần).
- Loại bỏ các cụm từ trùng lặp (Ví dụ: "PHÔNG PHÔNG PHÔNG...").
- TUYỆT ĐỐI KHÔNG sửa tên riêng gốc nước ngoài thành các từ Việt hóa như "Renan Gôm"
- GIỮ NGUYÊN toàn bộ đoạn văn là MỘT khối văn bản liền mạch như input, KHÔNG được tách thành nhiều dòng, nhiều câu riêng biệt, KHÔNG thêm dấu '|' hay bất kỳ dấu phân cách nào.
- TUYỆT ĐỐI KHÔNG thêm lời dẫn, lời chào, lời giải thích. Chỉ trả về đúng nội dung đoạn văn đã sửa, không có gì khác trước hay sau.

Đoạn văn:
{text}"""


def fix_texts_in_batches(llm, texts, batch_size, log_prefix=""):
    """Chunk 'texts' theo batch_size, gọi LLM sửa lỗi chính tả cho từng cụm,
    validate số lượng câu trả về, retry khi lỗi, fallback giữ nguyên text gốc
    của cụm nếu hết retry. Trả về list text đã sửa, cùng độ dài với 'texts'."""
    all_fixed_texts = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        is_single = len(batch) == 1
        prompt = build_single_prompt(batch[0]) if is_single else build_multi_prompt(batch)

        max_retries = 3
        retry_count = 0
        success = False
        fixed_texts = []

        while retry_count < max_retries and not success:
            try:
                response_text = llm.prompt(prompt)

                if is_single:
                    fixed_text = response_text.strip()
                    if not fixed_text:
                        raise ValueError("AI trả về rỗng.")
                    fixed_texts = [fixed_text]
                else:
                    fixed_texts = [line.strip() for line in response_text.split('|') if line.strip()]

                    if len(fixed_texts) != len(batch):
                        print(f"    ⚠️ {log_prefix}Cảnh báo cụm {i}: AI trả về {len(fixed_texts)} câu thay vì {len(batch)} câu.")
                        raise ValueError("Số lượng câu trả về từ AI không khớp với dữ liệu gốc.")

                success = True
                time.sleep(10.0)  # Nghỉ nhẹ tránh spam API

            except Exception as e:
                retry_count += 1
                print(f"    ⚠️ {log_prefix}Lỗi cụm {i} (Lần thử {retry_count}/{max_retries}): {e}")
                if retry_count < max_retries:
                    print("    💤 Server quá tải hoặc phản hồi lỗi. Ngủ đông 20 giây...")
                    time.sleep(20.0)
                else:
                    print(f"    ❌ {log_prefix}Đã thử hết cách. HỦY KẾT QUẢ CỤM NÀY, giữ nguyên toàn bộ text gốc của cụm!")
                    fixed_texts = list(batch)
                    print(f"    📄 {log_prefix}Text gốc (không sửa được) của cụm {i}:")
                    for t in fixed_texts:
                        print(f"        - {t}")

        all_fixed_texts.extend(fixed_texts)

    return all_fixed_texts

In [ ]:
DEFAULT_BATCH_SIZE = 5
SEG_BATCH_SIZE = 10       # Batch size khi phải tách xuống cấp segment
DEFAULT_DURATION = 100    # Ngưỡng (giây): event dài hơn ngưỡng này thì tách xuống segment để sửa

@kbench.task(name=f"Fix Transcripts", description=f"Càn quét sạch lỗi chính tả")
def fix_mega_transcripts_task(llm) -> None:

    # Source transcripts đã được group event từ bước trước, 
    # mỗi event có field "text" đã gộp sẵn từ các segment.
    base_input_dir = f"./grouped_event"
    base_output_dir = f"./fixed_video_transcripts"

    target_path = f"{base_input_dir}/transcripts"

    all_json_paths = []
    if os.path.exists(target_path):
        for file in os.listdir(target_path):
            if file.endswith(".json"):
                all_json_paths.append(os.path.join(target_path, file))
    else:
        print(f"❌ Không tìm thấy thư mục: {target_path}. Hãy kiểm tra lại LEVEL_NUMBER!")
        return

    all_json_paths.sort()
    total_files = len(all_json_paths)

    print(f"Starting fix_mega_transcripts_task. Tìm thấy tổng cộng {total_files} file JSON.")
    print("----------------------------------------------------------------------")

    for idx, input_path in enumerate(all_json_paths, 1):
        relative_path = os.path.relpath(input_path, base_input_dir)
        output_path = os.path.join(base_output_dir, relative_path)

        os.makedirs(os.path.dirname(output_path), exist_ok=True)

        print(f"    [{idx}/{total_files}] Đang xử lý: {relative_path}")

        try:
            with open(input_path, "r", encoding="utf-8") as f:
                transcript = json.load(f)
        except Exception as e:
            print(f"    ❌ Lỗi đọc file: {e}. Bỏ qua file này.")
            continue

        # Xử lý ở cấp EVENT: mỗi event có "text" đã gộp sẵn từ bước group trước đó.
        all_events = transcript.get('events', [])
        total_events = len(all_events)

        if total_events == 0:
            print(f"    ⏩ File trống (0 events). Ghi nhận và copy nguyên bản sang output.")
            # Ghi file gốc ra thư mục mới trước khi bỏ qua
            with open(output_path, "w", encoding="utf-8") as f:
                json.dump(transcript, f, ensure_ascii=False, indent=2)
            continue

        print(f"    📦 Có {total_events} events. Bắt đầu chia cụm...")

        if total_events <= 10:
            # 🔧 Ít event thường đồng nghĩa với event DÀI (do gom nhiều segment).
            # Prompt quá dài -> AI dễ sửa sai/thiếu. Nên xử lý từng event một,
            # và nếu event đó dài hơn DEFAULT_DURATION giây, tách tiếp xuống
            # cấp segment (đệ quy 1 tầng: unit lớn -> unit nhỏ hơn) để mỗi lần
            # gọi API chỉ xử lý đoạn text ngắn, dễ sửa chính xác hơn.
            for ev in all_events:
                duration = ev.get("end", 0) - ev.get("start", 0)

                if duration > DEFAULT_DURATION and ev.get("segments"):
                    seg_texts = [s["text"] for s in ev["segments"]]
                    fixed_seg_texts = fix_texts_in_batches(
                        llm, seg_texts, SEG_BATCH_SIZE,
                        log_prefix=f"[event_id={ev.get('event_id')}, seg] "
                    )
                    fixed_seg_texts = [normalize_segment_text(t) for t in fixed_seg_texts]
                    ev["text"] = " ".join(fixed_seg_texts)
                else:
                    fixed = fix_texts_in_batches(
                        llm, [ev["text"]], 1,
                        log_prefix=f"[event_id={ev.get('event_id')}] "
                    )
                    ev["text"] = fixed[0]
        else:
            # Nhiều event (thường ngắn hơn) -> gom theo DEFAULT_BATCH_SIZE.
            event_texts = [ev["text"] for ev in all_events]
            fixed_texts = fix_texts_in_batches(llm, event_texts, DEFAULT_BATCH_SIZE)
            for ev, fixed_text in zip(all_events, fixed_texts):
                ev["text"] = fixed_text

        # Xóa field "segments" và "words" trong từng event: "segments" không
        # được map ngược text đã sửa lỗi chính tả nên giữ lại chỉ gây hiểu nhầm
        # là đã đồng bộ (dù ở nhánh duration>DEFAULT_DURATION có DÙNG segments
        # để tính lại text, nhưng segments cũ vẫn chưa tự sửa chính tả từng
        # câu nên vẫn bỏ, chỉ giữ text tổng đã sửa ở event)
        for ev in all_events:
            ev.pop("segments", None)

        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(transcript, f, ensure_ascii=False, indent=2)

        print(f"    ✅ Đã lưu file sạch.")

    print(f"Fixing transcripts task completed. Tất cả {total_files} file JSON đã được xử lý và lưu vào thư mục output.")
    kbench.assertions.assert_true(True, expectation=f"Done")

# Run the task with a specific LLM
fix_mega_transcripts_task.run(llm=kbench.llms["google/gemini-3.5-flash"])